# NB-02: JD Rule Extraction — Hard Disqualifiers & Soft Negatives

**Goal:** Encode the JD's explicit hard-disqualifier and soft-negative rules as functions operating on real schema fields, building on NB-01's cleaned data. This notebook is the "secret sauce" we'll defend at Stage 5 — every rule must trace directly back to the JD's own language.

**Input:** `cleaned_candidates_v1.parquet` (from NB-01), raw `df` (career_history, skills — still needed for rules not yet flattened)

**Output:** `jd_rule_flags.parquet` (candidate_id + all hard-disqualifier and soft-negative boolean/score columns)

**Hard disqualifiers (verbatim from JD):**
1. Pure research/academic-only career, zero production deployment
2. "AI experience" = only LangChain-calls-OpenAI projects under 12 months old, with no pre-LLM-era ML production experience
3. Senior engineer, no production code written in 18+ months
4. Entire career at consulting firms, zero product-company experience
5. CV/speech/robotics-only career, no NLP/IR exposure
6. 5+ years entirely on closed-source proprietary systems, zero external validation

**Soft negatives:**
1. Title-chasing pattern (Senior→Staff→Principal via ~1.5yr company-hops)
2. "Framework enthusiast" signature (tutorial-style LangChain/hot-framework projects, not systems work)

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

BASE = Path("/kaggle/input/datasets/kritikabenjwal/india-runs-datset/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge")
NB01_OUT = Path("/kaggle/input/notebooks/kritikabenjwal/eda-and-schema-audit")  # adjust once you know the actual Kaggle dataset path for NB-01's saved output

# Load cleaned flattened data from NB-01
df_flat = pd.read_parquet(NB01_OUT / "cleaned_candidates_v1.parquet")
print("df_flat shape:", df_flat.shape)

# Reload raw candidates.jsonl for full nested career_history/skills access
# (career_history and skills ARE preserved in df_flat as nested objects, so check first before reloading)
print("\ncareer_history dtype in df_flat:", df_flat['career_history'].dtype)
print("Sample career_history[0]:", df_flat['career_history'].iloc[0])
print("\nskills dtype in df_flat:", df_flat['skills'].dtype)
print("Sample skills[0]:", df_flat['skills'].iloc[0])

df_flat shape: (100000, 51)

career_history dtype in df_flat: object
Sample career_history[0]: [{'company': 'Mindtree', 'title': 'Backend Engineer', 'start_date': '2024-03-08', 'end_date': None, 'duration_months': 27, 'is_current': True, 'industry': 'IT Services', 'company_size': '10001+', 'description': 'Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.'}
 {'company': 'Dunder Mifflin', 'title': 'Analytics Engineer', 'start_date': '2019-07-03', 'end_date': '2024-01-08', 'duration_months': 55, 'is_current': False, 'industry': 'Paper Products', 'company_size': '201-500', 'description': 'Built and maint

## Step 1: Hard Disqualifier #4 (consulting-only) & #5 (CV/speech/robotics-only, no NLP/IR)

Rule 4 was already built in NB-01 as `flag_consulting_only_career` — reuse it here for continuity.

Rule 5: candidate's entire skill/experience footprint is Computer Vision, Speech, or Robotics with **zero** NLP/IR-adjacent skills or experience. Per the JD, a Senior AI Engineer for a *search/ranking* role needs some NLP/IR exposure — pure CV/robotics/speech specialists don't transfer.

In [2]:
CONSULTING_FIRMS = {'TCS', 'Infosys', 'Wipro', 'Accenture', 'Cognizant', 'Capgemini'}

def is_consulting_only_career(career_history):
    companies = {job['company'] for job in career_history}
    return companies.issubset(CONSULTING_FIRMS)

df_flat['hard_disq_consulting_only'] = df_flat['career_history'].apply(is_consulting_only_career)
print("Hard disqualifier — consulting-only career:", df_flat['hard_disq_consulting_only'].sum())

# Skill taxonomy buckets (built from what we've seen in the data so far — CV, Speech, Robotics vs NLP/IR)
CV_SKILLS = {'Image Classification', 'Object Detection', 'GANs', 'Computer Vision'}
SPEECH_SKILLS = {'Speech Recognition', 'TTS'}
ROBOTICS_SKILLS = {'Robotics', 'ROS', 'Control Systems'}
NLP_IR_SKILLS = {'NLP', 'Embeddings', 'Semantic Search', 'Vector Search', 'Retrieval',
                  'RAG', 'Information Retrieval', 'LlamaIndex', 'LangChain', 'Ranking',
                  'Recommendation Systems', 'Search'}

CV_SPEECH_ROBOTICS = CV_SKILLS | SPEECH_SKILLS | ROBOTICS_SKILLS

def is_cv_speech_robotics_only(skills_list):
    skill_names = {s['name'] for s in skills_list}
    has_cv_speech_robotics = bool(skill_names & CV_SPEECH_ROBOTICS)
    has_nlp_ir = bool(skill_names & NLP_IR_SKILLS)
    return has_cv_speech_robotics and not has_nlp_ir

df_flat['hard_disq_cv_speech_robotics_only'] = df_flat['skills'].apply(is_cv_speech_robotics_only)
print("Hard disqualifier — CV/speech/robotics-only, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum())

# Peek: full set of unique skill names in the dataset -- so we can verify our taxonomy buckets are complete
all_skill_names = set()
for skills_list in df_flat['skills']:
    all_skill_names.update(s['name'] for s in skills_list)
print(f"\nTotal unique skill names in dataset: {len(all_skill_names)}")
print(sorted(all_skill_names))

Hard disqualifier — consulting-only career: 7034
Hard disqualifier — CV/speech/robotics-only, no NLP/IR: 17219

Total unique skill names in dataset: 133
['ASR', 'AWS', 'Accounting', 'Agile', 'Airflow', 'Angular', 'Apache Beam', 'Apache Flink', 'Azure', 'BM25', 'BentoML', 'BigQuery', 'CI/CD', 'CNN', 'CSS', 'Computer Vision', 'Content Matching', 'Content Writing', 'Data Pipelines', 'Data Science', 'Databricks', 'Deep Learning', 'Diffusion Models', 'Django', 'Docker', 'Document Processing', 'ETL', 'Elasticsearch', 'Embeddings', 'Excel', 'FAISS', 'FastAPI', 'Feature Engineering', 'Figma', 'Fine-tuning LLMs', 'Flask', 'Forecasting', 'GANs', 'GCP', 'Go', 'GraphQL', 'HTML', 'Hadoop', 'Haystack', 'Hugging Face Transformers', 'Illustrator', 'Image Classification', 'Indexing Algorithms', 'Information Retrieval', 'Information Retrieval Systems', 'Java', 'JavaScript', 'Kafka', 'Kubeflow', 'Kubernetes', 'LLMs', 'LangChain', 'Learning to Rank', 'LlamaIndex', 'LoRA', 'MLOps', 'MLflow', 'Machine Learn

## Step 1b: Corrected skill taxonomy — full vocabulary coverage

Original taxonomy only covered a handful of skills and missed most of the real NLP/IR/search vocabulary (vector DBs, search infra, ranking terms, transformer/embedding terms). Rebuilt against the actual 133 unique skill names.

In [3]:
CV_SKILLS = {'Computer Vision', 'Image Classification', 'Object Detection', 'CNN', 'YOLO',
             'OpenCV', 'GANs', 'Diffusion Models'}
SPEECH_SKILLS = {'Speech Recognition', 'TTS', 'ASR'}
ROBOTICS_SKILLS = set()  # none present in this dataset's vocabulary — kept as empty set for completeness/documentation

NLP_IR_SKILLS = {
    'NLP', 'Natural Language Processing', 'Information Retrieval', 'Information Retrieval Systems',
    'Embeddings', 'Semantic Search', 'Vector Search', 'Vector Representations', 'RAG',
    'LangChain', 'LlamaIndex', 'Ranking Systems', 'Learning to Rank', 'Recommendation Systems',
    'Search & Discovery', 'Search Backend', 'Search Infrastructure', 'Sentence Transformers',
    'Text Encoders', 'BM25', 'FAISS', 'Haystack', 'Elasticsearch', 'OpenSearch', 'Pinecone',
    'Qdrant', 'Weaviate', 'Milvus', 'pgvector', 'Hugging Face Transformers', 'Content Matching',
    'LLMs', 'Fine-tuning LLMs', 'Prompt Engineering', 'Indexing Algorithms'
}

CV_SPEECH_ROBOTICS = CV_SKILLS | SPEECH_SKILLS | ROBOTICS_SKILLS

# Sanity check: confirm every skill in the dataset is categorized somewhere (or intentionally left general/unrelated)
GENERAL_ML_INFRA = {'Deep Learning', 'Machine Learning', 'Reinforcement Learning', 'PyTorch', 'TensorFlow',
                     'scikit-learn', 'MLOps', 'MLflow', 'Kubeflow', 'Feature Engineering', 'Statistical Modeling',
                     'Time Series', 'Forecasting', 'Model Adaptation', 'LoRA', 'QLoRA', 'PEFT',
                     'Open-source ML libraries', 'Data Science'}
NON_ML = {'AWS', 'Accounting', 'Agile', 'Airflow', 'Angular', 'Apache Beam', 'Apache Flink', 'Azure',
          'BentoML', 'BigQuery', 'CI/CD', 'CSS', 'Content Writing', 'Data Pipelines', 'Databricks',
          'Django', 'Docker', 'Document Processing', 'ETL', 'Excel', 'FastAPI', 'Figma', 'Flask',
          'GCP', 'Go', 'GraphQL', 'HTML', 'Hadoop', 'Illustrator', 'Java', 'JavaScript', 'Kafka',
          'Kubernetes', 'Marketing', 'Microservices', 'MongoDB', 'Next.js', 'Node.js', 'PostgreSQL',
          'PowerPoint', 'Project Management', 'Python', 'REST APIs', 'React', 'Redis', 'Redux',
          'Rust', 'SAP', 'SEO', 'SQL', 'Sales', 'Salesforce CRM', 'Scrum', 'Six Sigma', 'Snowflake',
          'Spark', 'Spring Boot', 'Tailwind', 'Tally', 'Terraform', 'TypeScript', 'Vue.js', 'Webpack',
          'Weights & Biases', 'Workflow Orchestration', 'dbt', 'gRPC'}

categorized = CV_SPEECH_ROBOTICS | NLP_IR_SKILLS | GENERAL_ML_INFRA | NON_ML
uncategorized = set(all_skill_names) - categorized
print("Uncategorized skills (should be empty or near-empty):", uncategorized)

def is_cv_speech_robotics_only(skills_list):
    skill_names = {s['name'] for s in skills_list}
    has_cv_speech_robotics = bool(skill_names & CV_SPEECH_ROBOTICS)
    has_nlp_ir = bool(skill_names & NLP_IR_SKILLS)
    return has_cv_speech_robotics and not has_nlp_ir

df_flat['hard_disq_cv_speech_robotics_only'] = df_flat['skills'].apply(is_cv_speech_robotics_only)
print("\nHard disqualifier — CV/speech/robotics-only, no NLP/IR (corrected):", df_flat['hard_disq_cv_speech_robotics_only'].sum())

# Sample check
sample_flagged = df_flat[df_flat['hard_disq_cv_speech_robotics_only']].head(5)
print("\n--- Sample flagged candidates ---")
for idx, row in sample_flagged.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']}: {skill_names}")

Uncategorized skills (should be empty or near-empty): {'Photoshop'}

Hard disqualifier — CV/speech/robotics-only, no NLP/IR (corrected): 22260

--- Sample flagged candidates ---
CAND_0000004: ['Node.js', 'Content Writing', 'Redux', 'Airflow', 'GraphQL', 'Object Detection', 'Webpack', 'Six Sigma', 'SAP', 'JavaScript']
CAND_0000005: ['SQL', 'PowerPoint', 'Photoshop', 'Tailwind', 'Apache Flink', 'Image Classification']
CAND_0000009: ['Snowflake', 'gRPC', 'JavaScript', 'OpenCV', 'Go', 'PowerPoint']
CAND_0000018: ['CNN', 'Java', 'Accounting', 'Data Pipelines', 'Node.js', 'Tailwind']
CAND_0000019: ['Figma', 'GraphQL', 'Six Sigma', 'Scrum', 'YOLO', 'gRPC', 'AWS', 'Azure']


## Step 1c: Fix — rule 5 was catching non-ML candidates with one stray CV-adjacent skill

The JD's intent is "genuine ML engineer, but CV/speech/robotics specialist with no NLP/IR" — not "anyone lacking NLP skills." Gate the rule behind a minimum ML/AI skill presence first.

In [4]:
NON_ML = NON_ML | {'Photoshop'}  # fix uncategorized skill

ML_AI_SKILLS = CV_SPEECH_ROBOTICS | NLP_IR_SKILLS | GENERAL_ML_INFRA

def is_cv_speech_robotics_only(skills_list, min_ml_skills=3):
    skill_names = {s['name'] for s in skills_list}
    ml_ai_skill_count = len(skill_names & ML_AI_SKILLS)

    # Gate: must actually be an ML/AI-oriented profile first (not a web dev with one stray CV skill)
    if ml_ai_skill_count < min_ml_skills:
        return False

    has_cv_speech_robotics = bool(skill_names & CV_SPEECH_ROBOTICS)
    has_nlp_ir = bool(skill_names & NLP_IR_SKILLS)
    return has_cv_speech_robotics and not has_nlp_ir

df_flat['hard_disq_cv_speech_robotics_only'] = df_flat['skills'].apply(is_cv_speech_robotics_only)
print("Hard disqualifier — CV/speech/robotics-only, genuine ML profile, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum())

# Re-check samples
sample_flagged = df_flat[df_flat['hard_disq_cv_speech_robotics_only']].head(8)
print("\n--- Sample flagged candidates (should now look like real CV/speech ML specialists) ---")
for idx, row in sample_flagged.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} ({row['profile_current_title']}): {skill_names}")

# Confirm uncategorized is now empty
uncategorized = set(all_skill_names) - (CV_SPEECH_ROBOTICS | NLP_IR_SKILLS | GENERAL_ML_INFRA | NON_ML)
print("\nUncategorized skills (should be empty now):", uncategorized)

Hard disqualifier — CV/speech/robotics-only, genuine ML profile, no NLP/IR: 2907

--- Sample flagged candidates (should now look like real CV/speech ML specialists) ---
CAND_0000051 (Cloud Engineer): ['Databricks', 'MongoDB', 'YOLO', 'Diffusion Models', 'Forecasting', 'PostgreSQL', 'REST APIs', 'Accounting', 'SAP', 'React']
CAND_0000059 (Frontend Engineer): ['Six Sigma', 'TypeScript', 'GANs', 'Azure', 'Statistical Modeling', 'HTML', 'Rust', 'gRPC', 'Agile', 'CNN', 'Sales', 'Terraform', 'CSS']
CAND_0000061 (DevOps Engineer): ['Illustrator', 'Speech Recognition', 'JavaScript', 'Airflow', 'TypeScript', 'Sales', 'TTS', 'Object Detection', 'Spark']
CAND_0000090 (QA Engineer): ['YOLO', 'Apache Beam', 'Apache Flink', 'Rust', 'Angular', 'AWS', 'Redux', 'Databricks', 'ETL', 'Figma', 'MLOps', 'Forecasting', 'SAP', 'Docker', 'Vue.js']
CAND_0000152 (QA Engineer): ['PowerPoint', 'Kubeflow', 'Airflow', 'SQL', 'Scrum', 'Diffusion Models', 'Databricks', 'YOLO', 'AWS', 'Snowflake', 'Redis']
CAND_000018

## Step 1d: Fix — gate on title/career evidence of AI/ML role, not just skill-tag count

Skills appear to be assigned somewhat independently of career title in this synthetic dataset. Skill-count alone isn't a reliable "is this person actually an ML engineer" signal. Gate on `current_title` / career history titles containing ML/AI/Data-Science-adjacent language instead.

In [5]:
AI_ML_TITLE_KEYWORDS = ['ml engineer', 'machine learning', 'ai engineer', 'data scientist',
                         'research scientist', 'nlp engineer', 'computer vision engineer',
                         'applied scientist', 'deep learning', 'ai researcher']

def has_ml_ai_title(current_title, career_history):
    all_titles = [current_title] + [job['title'] for job in career_history]
    all_titles_lower = ' | '.join(all_titles).lower()
    return any(kw in all_titles_lower for kw in AI_ML_TITLE_KEYWORDS)

df_flat['is_ml_ai_practitioner_by_title'] = df_flat.apply(
    lambda row: has_ml_ai_title(row['profile_current_title'], row['career_history']), axis=1
)
print("Candidates with an ML/AI-titled role anywhere in career history:", df_flat['is_ml_ai_practitioner_by_title'].sum())

def is_cv_speech_robotics_only(row, min_ml_skills=2):
    if not row['is_ml_ai_practitioner_by_title']:
        return False
    skill_names = {s['name'] for s in row['skills']}
    ml_ai_skill_count = len(skill_names & ML_AI_SKILLS)
    if ml_ai_skill_count < min_ml_skills:
        return False
    has_cv_speech_robotics = bool(skill_names & CV_SPEECH_ROBOTICS)
    has_nlp_ir = bool(skill_names & NLP_IR_SKILLS)
    return has_cv_speech_robotics and not has_nlp_ir

df_flat['hard_disq_cv_speech_robotics_only'] = df_flat.apply(is_cv_speech_robotics_only, axis=1)
print("Hard disqualifier — CV/speech/robotics-only, genuine ML/AI-titled career, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum())

sample_flagged = df_flat[df_flat['hard_disq_cv_speech_robotics_only']].head(8)
print("\n--- Sample flagged candidates ---")
for idx, row in sample_flagged.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
    print(f"{row['candidate_id']} | titles: {titles} | skills: {skill_names}")

Candidates with an ML/AI-titled role anywhere in career history: 988
Hard disqualifier — CV/speech/robotics-only, genuine ML/AI-titled career, no NLP/IR: 0

--- Sample flagged candidates ---


## Step 1e: Inspect the 988 ML/AI-titled candidates directly before re-tuning

Rather than keep guessing thresholds, look at what genuine ML/AI-titled candidates' skill profiles actually look like, so rule 5's logic is built from real data patterns, not assumptions.

In [6]:
ml_titled = df_flat[df_flat['is_ml_ai_practitioner_by_title']].copy()

# For each, compute CV/speech overlap and NLP/IR overlap
def skill_overlap_counts(skills_list):
    skill_names = {s['name'] for s in skills_list}
    return pd.Series({
        'cv_speech_count': len(skill_names & CV_SPEECH_ROBOTICS),
        'nlp_ir_count': len(skill_names & NLP_IR_SKILLS),
        'total_skills': len(skill_names)
    })

overlap_df = ml_titled['skills'].apply(skill_overlap_counts)
ml_titled = pd.concat([ml_titled, overlap_df], axis=1)

print("--- Distribution: cv_speech_count vs nlp_ir_count among ML/AI-titled candidates ---")
print(ml_titled[['cv_speech_count', 'nlp_ir_count', 'total_skills']].describe())

print("\n--- Cross-tab: has CV/speech skill vs has NLP/IR skill ---")
ml_titled['has_cv_speech'] = ml_titled['cv_speech_count'] > 0
ml_titled['has_nlp_ir'] = ml_titled['nlp_ir_count'] > 0
print(pd.crosstab(ml_titled['has_cv_speech'], ml_titled['has_nlp_ir']))

# The actual candidates matching "has CV/speech, zero NLP/IR" regardless of skill-count threshold
true_candidates = ml_titled[(ml_titled['cv_speech_count'] > 0) & (ml_titled['nlp_ir_count'] == 0)]
print(f"\nML/AI-titled + has CV/speech skill + zero NLP/IR skill: {len(true_candidates)}")
print("\n--- These candidates ---")
for idx, row in true_candidates.head(10).iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} | title: {row['profile_current_title']} | skills: {skill_names}")

--- Distribution: cv_speech_count vs nlp_ir_count among ML/AI-titled candidates ---
       cv_speech_count  nlp_ir_count  total_skills
count       988.000000    988.000000    988.000000
mean          2.868421      4.907895     14.794534
std           1.273570      1.946243      2.397079
min           0.000000      1.000000      9.000000
25%           2.000000      4.000000     13.000000
50%           3.000000      5.000000     15.000000
75%           4.000000      6.000000     16.000000
max           7.000000     12.000000     23.000000

--- Cross-tab: has CV/speech skill vs has NLP/IR skill ---
has_nlp_ir     True
has_cv_speech      
False            18
True            970

ML/AI-titled + has CV/speech skill + zero NLP/IR skill: 0

--- These candidates ---


## Step 1f: Finding — disqualifier #5 pattern does not occur in this dataset

Among all 988 genuinely ML/AI-titled candidates, every one has at least 1 NLP/IR-adjacent skill (min `nlp_ir_count` = 1). The "CV/speech/robotics-only, zero NLP/IR" trap profile the JD describes was not generated into this dataset's ML-titled population.

**Decision:** keep the rule function as correctly-scoped, defensible logic (it would catch a real instance if one existed) but document that it returns 0 flags on this dataset. This is worth stating explicitly in the Stage 5 defense — we checked for it deliberately, not that we missed it.

In [7]:
# Final, locked version of rule 5 (kept in the codebase, documented as dormant on this dataset)
def hard_disq_cv_speech_robotics_only(row, min_ml_skills=2):
    """Genuine ML/AI-titled career + CV/speech/robotics skill present + zero NLP/IR skill.
    Confirmed dormant on this dataset (0 matches) -- kept for correctness/defensibility."""
    if not row['is_ml_ai_practitioner_by_title']:
        return False
    skill_names = {s['name'] for s in row['skills']}
    if len(skill_names & ML_AI_SKILLS) < min_ml_skills:
        return False
    return bool(skill_names & CV_SPEECH_ROBOTICS) and not bool(skill_names & NLP_IR_SKILLS)

df_flat['hard_disq_cv_speech_robotics_only'] = df_flat.apply(hard_disq_cv_speech_robotics_only, axis=1)
print("Confirmed final count (expected 0):", df_flat['hard_disq_cv_speech_robotics_only'].sum())

print("\n=== Hard disqualifiers so far ===")
print("1. Consulting-only career:", df_flat['hard_disq_consulting_only'].sum())
print("5. CV/speech/robotics-only, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum(), "(dormant, documented)")

Confirmed final count (expected 0): 0

=== Hard disqualifiers so far ===
1. Consulting-only career: 7034
5. CV/speech/robotics-only, no NLP/IR: 0 (dormant, documented)


## Step 1g: Hard Disqualifier #2 — "LangChain-wrapper-only" AI experience

JD wording: a candidate whose only AI/ML exposure is recent (<12 months) LangChain/OpenAI-API-calling work, with **no pre-LLM-era ML production experience** (classic ML, MLOps, traditional NLP/CV work predating the LLM-wrapper trend). This is the "framework enthusiast" pattern taken to its disqualifying extreme — someone who has never actually built/trained/deployed a model, only called an API.

Operationalize as:
- Has LangChain and/or "LLM-wrapper" skill tags (LangChain, OpenAI API, Prompt Engineering, RAG-as-a-tool-call — not model-building)
- That exposure is recent: appears only in career_history entries with `duration_months <= 12` (their most recent/current role)
- Zero evidence of pre-LLM-era ML production skills anywhere in their skill set (classic ML/DL frameworks, MLOps, traditional NLP/CV — the "GENERAL_ML_INFRA" style skills we already categorized)

In [8]:
LLM_WRAPPER_SKILLS = {'LangChain', 'LlamaIndex', 'Prompt Engineering', 'RAG'}
PRE_LLM_ML_PRODUCTION_SKILLS = GENERAL_ML_INFRA | NLP_IR_SKILLS | CV_SPEECH_ROBOTICS  # actual model-building/deploying skills

# Skills that indicate real pre-LLM ML production work, EXCLUDING the wrapper-only tags themselves
PRE_LLM_ML_PRODUCTION_SKILLS = PRE_LLM_ML_PRODUCTION_SKILLS - LLM_WRAPPER_SKILLS

def hard_disq_langchain_wrapper_only(row, recency_months=12):
    skill_names = {s['name'] for s in row['skills']}

    has_llm_wrapper_skill = bool(skill_names & LLM_WRAPPER_SKILLS)
    if not has_llm_wrapper_skill:
        return False

    # Zero pre-LLM-era ML production skill anywhere in their profile -> real disqualifying case
    has_real_ml_production_skill = bool(skill_names & PRE_LLM_ML_PRODUCTION_SKILLS)
    if has_real_ml_production_skill:
        return False  # they DO have pre-LLM production experience -> not disqualified

    # Confirm the LLM-wrapper exposure itself is recent (<= recency_months), via career_history
    # Check the most recent job (is_current=True, or the one with the latest start_date)
    current_jobs = [j for j in row['career_history'] if j.get('is_current')]
    recent_job = current_jobs[0] if current_jobs else max(row['career_history'], key=lambda j: j['start_date'])
    is_recent_only = recent_job.get('duration_months', 999) <= recency_months

    return is_recent_only

df_flat['hard_disq_langchain_wrapper_only'] = df_flat.apply(hard_disq_langchain_wrapper_only, axis=1)
print("Hard disqualifier #2 — LangChain-wrapper-only, no pre-LLM ML production experience:", df_flat['hard_disq_langchain_wrapper_only'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_langchain_wrapper_only']].head(8)
for idx, row in sample.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    current_job = next((j for j in row['career_history'] if j.get('is_current')), row['career_history'][-1])
    print(f"{row['candidate_id']} | current: {current_job['title']} @ {current_job['company']} ({current_job['duration_months']}mo) | skills: {skill_names}")

Hard disqualifier #2 — LangChain-wrapper-only, no pre-LLM ML production experience: 6

--- Sample flagged candidates ---
CAND_0017916 | current: Software Engineer @ Initech (12mo) | skills: ['Spark', 'GraphQL', 'TypeScript', 'PowerPoint', 'Hadoop', 'Excel', 'Six Sigma', 'Python', 'SEO', 'Rust', 'HTML', 'Vue.js', 'LlamaIndex', 'Flask']
CAND_0045309 | current: QA Engineer @ Capgemini (12mo) | skills: ['Node.js', 'Accounting', 'Tally', 'gRPC', 'LlamaIndex', 'REST APIs', 'Snowflake', 'SAP']
CAND_0045623 | current: DevOps Engineer @ Tech Mahindra (12mo) | skills: ['CI/CD', 'Docker', 'ETL', 'AWS', 'Agile', 'Airflow', 'LlamaIndex', 'Tally', 'Redux', 'Data Pipelines', 'Accounting']
CAND_0070262 | current: Mobile Developer @ Swiggy (12mo) | skills: ['Node.js', 'Kubernetes', 'Django', 'LlamaIndex', 'Scrum', 'SAP', 'Tally', 'Microservices', 'JavaScript', 'AWS', 'Figma']
CAND_0092601 | current: DevOps Engineer @ Flipkart (12mo) | skills: ['GraphQL', 'LlamaIndex', 'Apache Beam', 'React', 'Excel', '

## Step 1h: Fix — gate LangChain-wrapper-only on candidates who actually present as AI/ML candidates

Same false-positive pattern as disqualifier #5: a stray `LangChain`/`RAG`/`LlamaIndex` skill tag on an unrelated profile (QA, DevOps, Mobile) isn't "claiming AI experience" — it's noise from this dataset's semi-randomized skill assignment. Gate on headline/summary language signaling an AI/ML self-presentation before applying the wrapper-only check.

In [9]:
AI_SELF_PRESENTATION_KEYWORDS = ['ai engineer', 'ai/ml', 'machine learning', 'llm', 'genai',
                                   'generative ai', 'nlp', 'ai researcher', 'ai practitioner',
                                   'artificial intelligence', 'deep learning']

def presents_as_ai_candidate(row):
    text = f"{row['profile_headline']} {row['profile_summary']} {row['profile_current_title']}".lower()
    return any(kw in text for kw in AI_SELF_PRESENTATION_KEYWORDS)

df_flat['presents_as_ai_candidate'] = df_flat.apply(presents_as_ai_candidate, axis=1)
print("Candidates presenting as AI/ML in headline/summary/title:", df_flat['presents_as_ai_candidate'].sum())

def hard_disq_langchain_wrapper_only(row, recency_months=12):
    if not row['presents_as_ai_candidate']:
        return False

    skill_names = {s['name'] for s in row['skills']}
    has_llm_wrapper_skill = bool(skill_names & LLM_WRAPPER_SKILLS)
    if not has_llm_wrapper_skill:
        return False

    has_real_ml_production_skill = bool(skill_names & PRE_LLM_ML_PRODUCTION_SKILLS)
    if has_real_ml_production_skill:
        return False

    current_jobs = [j for j in row['career_history'] if j.get('is_current')]
    recent_job = current_jobs[0] if current_jobs else max(row['career_history'], key=lambda j: j['start_date'])
    return recent_job.get('duration_months', 999) <= recency_months

df_flat['hard_disq_langchain_wrapper_only'] = df_flat.apply(hard_disq_langchain_wrapper_only, axis=1)
print("Hard disqualifier #2 (gated) — LangChain-wrapper-only, self-presents as AI candidate:", df_flat['hard_disq_langchain_wrapper_only'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_langchain_wrapper_only']].head(8)
for idx, row in sample.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    current_job = next((j for j in row['career_history'] if j.get('is_current')), row['career_history'][-1])
    print(f"{row['candidate_id']} | headline: {row['profile_headline']!r} | current: {current_job['title']} @ {current_job['company']} ({current_job['duration_months']}mo) | skills: {skill_names}")

Candidates presenting as AI/ML in headline/summary/title: 36694
Hard disqualifier #2 (gated) — LangChain-wrapper-only, self-presents as AI candidate: 6

--- Sample flagged candidates ---
CAND_0017916 | headline: 'Software Engineer | Backend systems & APIs' | current: Software Engineer @ Initech (12mo) | skills: ['Spark', 'GraphQL', 'TypeScript', 'PowerPoint', 'Hadoop', 'Excel', 'Six Sigma', 'Python', 'SEO', 'Rust', 'HTML', 'Vue.js', 'LlamaIndex', 'Flask']
CAND_0045309 | headline: 'QA Engineer | Cloud & DevOps' | current: QA Engineer @ Capgemini (12mo) | skills: ['Node.js', 'Accounting', 'Tally', 'gRPC', 'LlamaIndex', 'REST APIs', 'Snowflake', 'SAP']
CAND_0045623 | headline: 'DevOps Engineer | Full-stack development' | current: DevOps Engineer @ Tech Mahindra (12mo) | skills: ['CI/CD', 'Docker', 'ETL', 'AWS', 'Agile', 'Airflow', 'LlamaIndex', 'Tally', 'Redux', 'Data Pipelines', 'Accounting']
CAND_0070262 | headline: 'Mobile Developer | Full-stack development' | current: Mobile Developer

## Step 1i: Debug — why did the AI self-presentation gate not filter these out?

None of the flagged headlines mention AI/ML, yet they passed the `presents_as_ai_candidate` gate. The match is likely coming from generic aspirational language in `summary` (e.g. "curious about AI tools", "experimented with ChatGPT") rather than genuine AI/ML practitioner framing. Print the actual summary text to confirm before deciding how to fix the gate.

In [10]:
sample = df_flat[df_flat['hard_disq_langchain_wrapper_only']]

print("--- Full summary text for each flagged candidate ---")
for idx, row in sample.iterrows():
    print(f"\n{row['candidate_id']} | headline: {row['profile_headline']!r}")
    print(f"summary: {row['profile_summary']!r}")
    matched_keywords = [kw for kw in AI_SELF_PRESENTATION_KEYWORDS if kw in row['profile_summary'].lower()]
    print(f"matched keywords in summary: {matched_keywords}")

--- Full summary text for each flagged candidate ---

CAND_0017916 | headline: 'Software Engineer | Backend systems & APIs'
summary: "Software engineer with 7.1 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering — APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level — taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project — but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems."
matched keywords in summary: ['ai/ml']

CAND_0045309 | headline: 'QA Engineer | Cloud & DevOps'
summary: "Software engineer with 7.5 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My backgr

## Step 1j: Confirmed — disqualifier #2 rule is correct, not a false positive

The matched summaries are a consistent templated archetype: "self-learner AI/ML, played with OpenAI/Anthropic APIs, built a small RAG side project, haven't done it professionally." This is an exact match to the JD's stated disqualifier #2. Unlike disqualifier #5 (which was genuinely dormant/absent from the data), this pattern **does exist** — just as a small, deliberately rare templated population (6 candidates). Lock in the rule as-is, no further gating needed.

In [11]:
print("=== Hard disqualifiers — running tally ===")
print("1. Consulting-only career:", df_flat['hard_disq_consulting_only'].sum())
print("2. LangChain-wrapper-only, self-learner, no production ML:", df_flat['hard_disq_langchain_wrapper_only'].sum())
print("5. CV/speech/robotics-only, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum(), "(dormant, documented)")

print("\nRemaining to build: #1 pure-research-only (zero production deployment), "
      "#3 senior engineer with no production code in 18+ months (pure architecture/tech-lead), "
      "#6 closed-source-only 5+ yrs with zero external validation (no papers/talks/OSS)")

=== Hard disqualifiers — running tally ===
1. Consulting-only career: 7034
2. LangChain-wrapper-only, self-learner, no production ML: 6
5. CV/speech/robotics-only, no NLP/IR: 0 (dormant, documented)

Remaining to build: #1 pure-research-only (zero production deployment), #3 senior engineer with no production code in 18+ months (pure architecture/tech-lead), #6 closed-source-only 5+ yrs with zero external validation (no papers/talks/OSS)


## Step 1k: Hard Disqualifier #3 — Senior engineer, no production code in 18+ months (pure architecture/tech-lead)

JD wording: a senior-level candidate whose current role (18+ months tenure) is described purely in architecture/strategy/roadmap/stakeholder language, with no evidence of hands-on coding, implementation, or shipping — i.e., someone who has drifted fully into non-technical leadership and stopped being a practicing engineer.

Operationalize as:
- Seniority: current title contains a senior-level marker (senior, staff, principal, lead, architect, head of, director)
- Tenure gate: current role `duration_months >= 18`
- Description language: current role's `description` text contains architecture/strategy-only phrasing (e.g. "architected", "roadmap", "stakeholder", "led the strategy", "oversaw") **and** contains none of the hands-on/production coding phrasing (e.g. "built", "implemented", "shipped", "deployed", "wrote", "coded", "developed")

In [12]:
SENIORITY_TITLE_MARKERS = ['senior', 'staff', 'principal', 'lead', 'architect', 'head of', 'director']

ARCHITECTURE_ONLY_PHRASES = ['architected', 'roadmap', 'stakeholder', 'strategy', 'oversaw',
                              'overseeing', 'vision', 'governance', 'high-level design']

HANDS_ON_CODING_PHRASES = ['built', 'implemented', 'shipped', 'deployed', 'wrote', 'coded',
                             'developed', 'debugged', 'refactored', 'optimized', 'wrote code',
                             'hands-on', 'programmed']

def is_senior_title(title):
    return any(marker in title.lower() for marker in SENIORITY_TITLE_MARKERS)

def hard_disq_no_code_18mo_senior(row, tenure_months=18):
    current_jobs = [j for j in row['career_history'] if j.get('is_current')]
    if not current_jobs:
        return False
    current_job = current_jobs[0]

    if not is_senior_title(current_job['title']):
        return False
    if current_job.get('duration_months', 0) < tenure_months:
        return False

    desc = current_job.get('description', '').lower()
    has_architecture_language = any(p in desc for p in ARCHITECTURE_ONLY_PHRASES)
    has_hands_on_language = any(p in desc for p in HANDS_ON_CODING_PHRASES)

    return has_architecture_language and not has_hands_on_language

df_flat['hard_disq_no_code_18mo_senior'] = df_flat.apply(hard_disq_no_code_18mo_senior, axis=1)
print("Hard disqualifier #3 — senior, no production code in 18+ months:", df_flat['hard_disq_no_code_18mo_senior'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_no_code_18mo_senior']].head(8)
for idx, row in sample.iterrows():
    current_job = next(j for j in row['career_history'] if j.get('is_current'))
    print(f"\n{row['candidate_id']} | {current_job['title']} @ {current_job['company']} ({current_job['duration_months']}mo)")
    print(f"description: {current_job['description']!r}")

Hard disqualifier #3 — senior, no production code in 18+ months: 0

--- Sample flagged candidates ---


## Step 1l: Debug — funnel analysis for disqualifier #3 (0 flags)

Break the rule into its three gates (seniority, tenure, description language) and check the population passing each stage independently, to see where it collapses to zero.

In [13]:
def get_current_job(row):
    current_jobs = [j for j in row['career_history'] if j.get('is_current')]
    return current_jobs[0] if current_jobs else None

df_flat['_current_job'] = df_flat.apply(get_current_job, axis=1)
has_current = df_flat['_current_job'].notna()
print("Candidates with an is_current=True job:", has_current.sum(), "/", len(df_flat))

is_senior = df_flat['_current_job'].apply(lambda j: is_senior_title(j['title']) if j else False)
print("...and senior-titled:", (has_current & is_senior).sum())

tenure_ok = df_flat['_current_job'].apply(lambda j: j.get('duration_months', 0) >= 18 if j else False)
print("...and tenure >= 18mo:", (has_current & is_senior & tenure_ok).sum())

has_arch = df_flat['_current_job'].apply(lambda j: any(p in j.get('description','').lower() for p in ARCHITECTURE_ONLY_PHRASES) if j else False)
print("...and has architecture-language:", (has_current & is_senior & tenure_ok & has_arch).sum())

has_hands_on = df_flat['_current_job'].apply(lambda j: any(p in j.get('description','').lower() for p in HANDS_ON_CODING_PHRASES) if j else False)
print("...and ALSO has hands-on language (would disqualify from our rule):", (has_current & is_senior & tenure_ok & has_arch & has_hands_on).sum())

# Print a few actual descriptions from senior+tenured candidates regardless of phrase match, to see real language patterns
print("\n--- Sample descriptions from senior, 18mo+ tenured candidates ---")
sample_pool = df_flat[has_current & is_senior & tenure_ok].head(6)
for idx, row in sample_pool.iterrows():
    j = row['_current_job']
    print(f"\n{row['candidate_id']} | {j['title']} ({j['duration_months']}mo)")
    print(f"description: {j['description']!r}")

Candidates with an is_current=True job: 100000 / 100000
...and senior-titled: 1530
...and tenure >= 18mo: 1345
...and has architecture-language: 49
...and ALSO has hands-on language (would disqualify from our rule): 49

--- Sample descriptions from senior, 18mo+ tenured candidates ---

CAND_0000229 | Senior Software Engineer (26mo)
description: "Mixed data science and analytics-engineering role at a marketing-analytics startup. Spent maybe 30% of my time on lightweight ML (clustering, classification, churn prediction in sklearn/XGBoost) and 70% on data infrastructure and dashboards. Comfortable with the modeling work but I wouldn't call myself an ML specialist. Built our experimentation framework that supports the product team's A/B tests."

CAND_0000452 | Senior Data Engineer (19mo)
description: 'Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and 

## Step 1m: Inspect the 49 architecture-language candidates directly

Perfect overlap between architecture-language and hands-on-language (49 = 49) suggests our phrase lists are too generic to distinguish "pure architect" from "hands-on senior engineer who also mentions strategy/oversight." Read the actual descriptions before concluding the pattern doesn't exist.

In [14]:
arch_pool = df_flat[has_current & is_senior & tenure_ok & has_arch]
print(f"Inspecting all {len(arch_pool)} architecture-language candidates:\n")

for idx, row in arch_pool.iterrows():
    j = row['_current_job']
    matched_arch = [p for p in ARCHITECTURE_ONLY_PHRASES if p in j['description'].lower()]
    matched_hands_on = [p for p in HANDS_ON_CODING_PHRASES if p in j['description'].lower()]
    print(f"{row['candidate_id']} | {j['title']} ({j['duration_months']}mo)")
    print(f"  matched arch: {matched_arch} | matched hands-on: {matched_hands_on}")
    print(f"  description: {j['description']!r}\n")

Inspecting all 49 architecture-language candidates:

CAND_0001389 | Senior Software Engineer (ML) (37mo)
  matched arch: ['stakeholder'] | matched hands-on: ['built']
  description: "Worked on time-series forecasting models for supply-chain demand prediction at a logistics company. Built models in Prophet, LightGBM, and (for one project) a small LSTM — the LightGBM model ended up shipping. Also ran some reinforcement learning experiments for dynamic pricing but those didn't make it to production. The work was a mix of modeling, analysis, and stakeholder communication with the operations team."

CAND_0002344 | Senior Software Engineer (ML) (20mo)
  matched arch: ['vision'] | matched hands-on: ['built']
  description: "Built computer vision models for our product's image moderation feature using PyTorch — fine-tuned ResNet variants on a labeled dataset of ~200K images. Set up the training pipeline (data loading, augmentation, evaluation) and the inference service. Most of my project work

## Step 1n: Finding — disqualifier #3 does not occur; descriptions are drawn from a fixed template pool

The 49 "architecture-language" matches were false positives from generic words ("vision", "stakeholder") appearing inside genuinely hands-on descriptions. Notably, several candidates share byte-for-byte identical `description` text (e.g. the CV/PyTorch/ResNet description repeats verbatim across CAND_0002344, 0003756, 0005269, 0006821...). This strongly suggests `career_history.description` is sampled from a finite template bank rather than uniquely generated per candidate.

**Decision:** disqualifier #3 ("senior, no production code in 18+ months") is dormant on this dataset — confirm via template-uniqueness check, then lock the rule as correctly-scoped-but-dormant, same treatment as disqualifier #5.

In [15]:
# Confirm the template-pool hypothesis: how many unique description strings exist vs total career_history entries?
all_descriptions = []
for ch in df_flat['career_history']:
    for job in ch:
        all_descriptions.append(job['description'])

print(f"Total career_history entries: {len(all_descriptions)}")
print(f"Unique description strings: {len(set(all_descriptions))}")
print(f"Ratio (entries per unique template): {len(all_descriptions) / len(set(all_descriptions)):.1f}")

# Final: lock disqualifier #3 as dormant, matching the #5 treatment
def hard_disq_no_code_18mo_senior(row, tenure_months=18):
    """Senior title, 18+mo tenure, architecture-only language with zero hands-on coding language.
    Confirmed dormant on this dataset (descriptions are template-pooled; no pure-architect template exists)."""
    current_jobs = [j for j in row['career_history'] if j.get('is_current')]
    if not current_jobs:
        return False
    current_job = current_jobs[0]
    if not is_senior_title(current_job['title']) or current_job.get('duration_months', 0) < tenure_months:
        return False
    desc = current_job.get('description', '').lower()
    has_arch = any(p in desc for p in ARCHITECTURE_ONLY_PHRASES)
    has_hands_on = any(p in desc for p in HANDS_ON_CODING_PHRASES)
    return has_arch and not has_hands_on

df_flat['hard_disq_no_code_18mo_senior'] = df_flat.apply(hard_disq_no_code_18mo_senior, axis=1)

print("\n=== Hard disqualifiers — running tally ===")
print("1. Consulting-only career:", df_flat['hard_disq_consulting_only'].sum())
print("2. LangChain-wrapper-only, self-learner:", df_flat['hard_disq_langchain_wrapper_only'].sum())
print("3. No production code 18mo+ senior:", df_flat['hard_disq_no_code_18mo_senior'].sum(), "(dormant, documented)")
print("5. CV/speech/robotics-only, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum(), "(dormant, documented)")

Total career_history entries: 300171
Unique description strings: 44
Ratio (entries per unique template): 6822.1

=== Hard disqualifiers — running tally ===
1. Consulting-only career: 7034
2. LangChain-wrapper-only, self-learner: 6
3. No production code 18mo+ senior: 0 (dormant, documented)
5. CV/speech/robotics-only, no NLP/IR: 0 (dormant, documented)


## Step 1o: Strategy shift — read all 44 unique description templates directly

Confirmed: `career_history.description` is drawn from a fixed pool of only 44 templates (6,822 uses each on average), not freely generated text. This means we can classify every disqualifier pattern by direct inspection instead of keyword-guessing — a much more reliable approach for the remaining disqualifiers #1 (pure-research-only, zero production deployment) and #6 (closed-source-only 5+ yrs, zero external validation).

Print all 44 templates in full so we can manually tag which ones represent: research-only, closed-source-only, or neither.

In [16]:
unique_descriptions = sorted(set(all_descriptions))
print(f"Total unique templates: {len(unique_descriptions)}\n")

for i, desc in enumerate(unique_descriptions):
    print(f"--- TEMPLATE {i} ---")
    print(desc)
    print()

Total unique templates: 44

--- TEMPLATE 0 ---
Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering.

--- TEMPLATE 1 ---
Backend + data hybrid role at a growth-stage startup. Built the company's first proper data warehouse (migrating from a tangled set of Postgres replicas to a clean Snowflake setup with dbt), the orchestration layer (Airflow), and the BI integration (Looker). Shipped a couple of small predictive features but the bulk of the role was data infrastructure.

--- TEMPLATE 2 ---
Backend development with Python (FastAPI), PostgreSQL, and Redis at a B2B SaaS product. Owned t

## Step 1p: Disqualifier #1 (pure-research-only) — check titles directly, since none of the 44 templates match

None of the 44 career_history description templates describe a pure-research/academic-only profile with zero production deployment — every template involves applied, production-adjacent work. But titles aren't limited to these 44 templates, so check `current_title` and career_history titles for research/academic keywords across all 100K candidates before concluding this disqualifier is dormant.

In [17]:
RESEARCH_ACADEMIC_TITLE_KEYWORDS = ['research scientist', 'research engineer', 'phd', 'postdoc',
                                      'professor', 'academic', 'research fellow', 'doctoral']

def has_research_academic_title(row):
    all_titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
    text = ' | '.join(all_titles).lower()
    return any(kw in text for kw in RESEARCH_ACADEMIC_TITLE_KEYWORDS)

df_flat['_has_research_title'] = df_flat.apply(has_research_academic_title, axis=1)
print("Candidates with any research/academic-titled role in career history:", df_flat['_has_research_title'].sum())

if df_flat['_has_research_title'].sum() > 0:
    sample = df_flat[df_flat['_has_research_title']].head(10)
    print("\n--- Sample ---")
    for idx, row in sample.iterrows():
        titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
        print(f"{row['candidate_id']} | titles: {titles}")

df_flat = df_flat.drop(columns=['_has_research_title'])

Candidates with any research/academic-titled role in career history: 283

--- Sample ---
CAND_0000422 | titles: ['AI Research Engineer', 'AI Research Engineer', 'Data Scientist', 'Data Scientist']
CAND_0000666 | titles: ['Data Scientist', 'Data Scientist', 'AI Research Engineer', 'ML Engineer']
CAND_0000705 | titles: ['AI Research Engineer', 'AI Research Engineer', 'Junior ML Engineer']
CAND_0001302 | titles: ['Computer Vision Engineer', 'Computer Vision Engineer', 'AI Research Engineer', 'Data Scientist']
CAND_0001651 | titles: ['Data Scientist', 'Data Scientist', 'Computer Vision Engineer', 'AI Research Engineer']
CAND_0001808 | titles: ['ML Engineer', 'ML Engineer', 'AI Research Engineer']
CAND_0001940 | titles: ['AI Research Engineer', 'AI Research Engineer', 'Junior ML Engineer']
CAND_0002270 | titles: ['Computer Vision Engineer', 'Computer Vision Engineer', 'AI Research Engineer']
CAND_0002770 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer', 'AI 

## Step 1q: Fix — check for ENTIRE career being research/academic-titled, not just containing one research role

283 candidates mix "AI Research Engineer" with applied titles (Data Scientist, ML Engineer, CV Engineer) — that's a normal applied-ML trajectory, not the disqualifying pattern. The JD's disqualifier is "pure research/academic-only career" — mirror the consulting-only logic: ALL career_history titles must be research/academic, not just one.

In [18]:
def is_research_only_career(row):
    all_titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
    return all(any(kw in t.lower() for kw in RESEARCH_ACADEMIC_TITLE_KEYWORDS) for t in all_titles)

df_flat['hard_disq_research_only_career'] = df_flat.apply(is_research_only_career, axis=1)
print("Hard disqualifier #1 — entire career research/academic-titled, zero production evidence:", df_flat['hard_disq_research_only_career'].sum())

if df_flat['hard_disq_research_only_career'].sum() > 0:
    sample = df_flat[df_flat['hard_disq_research_only_career']].head(10)
    print("\n--- Sample ---")
    for idx, row in sample.iterrows():
        titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
        print(f"{row['candidate_id']} | titles: {titles}")

Hard disqualifier #1 — entire career research/academic-titled, zero production evidence: 32

--- Sample ---
CAND_0004628 | titles: ['AI Research Engineer', 'AI Research Engineer']
CAND_0010000 | titles: ['AI Research Engineer', 'AI Research Engineer']
CAND_0021491 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer']
CAND_0021827 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer']
CAND_0022415 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer']
CAND_0023076 | titles: ['AI Research Engineer', 'AI Research Engineer']
CAND_0023458 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer']
CAND_0024630 | titles: ['AI Research Engineer', 'AI Research Engineer']
CAND_0029294 | titles: ['AI Research Engineer', 'AI Research Engineer', 'AI Research Engineer']
CAND_0034849 | titles: ['AI Research Engineer', 'AI Research Engineer']


## Step 1r: Verify disqualifier #1 — check description text for the 32 flagged candidates

Title-only matching confirmed 32 candidates with an entirely research/academic-titled career. Before locking this in, verify their `career_history.description` text is consistent with "zero production deployment" — not just a research title layered on genuinely applied work.

In [19]:
research_only_pool = df_flat[df_flat['hard_disq_research_only_career']]

print(f"Inspecting all {len(research_only_pool)} research-only candidates:\n")

for idx, row in research_only_pool.iterrows():
    print(f"{row['candidate_id']} | yoe: {row['profile_years_of_experience']}")
    for j in row['career_history']:
        print(f"  {j['title']} @ {j['company']} ({j['duration_months']}mo): {j['description'][:200]}...")
    print()

Inspecting all 32 research-only candidates:

CAND_0004628 | yoe: 3.3
  AI Research Engineer @ PharmEasy (39mo): Contributed to ML feature engineering and model deployment for a fraud-detection product. My main role was engineering: building the Flask-based prediction API, integrating with the feature store, and...

CAND_0010000 | yoe: 3.2
  AI Research Engineer @ Aganitha (38mo): Worked on customer-facing predictive modeling for an e-commerce platform — churn prediction, conversion likelihood, lifetime value estimation. Used scikit-learn and XGBoost; main models were gradient-...

CAND_0021491 | yoe: 6.1
  AI Research Engineer @ Wysa (52mo): Worked on customer-facing predictive modeling for an e-commerce platform — churn prediction, conversion likelihood, lifetime value estimation. Used scikit-learn and XGBoost; main models were gradient-...
  AI Research Engineer @ Wipro (20mo): Built NLP pipelines for sentiment analysis and document classification — primarily for an internal feedback

## Step 1s: Finding — disqualifier #1 does not occur; "AI Research Engineer" title is not academic/research-only in this dataset

All 32 title-matched candidates have career_history descriptions drawn from the same 44-template pool, and every description involves explicit production deployment (Flask prediction APIs, feature-store integration, shipped CV models, customer-facing modeling). "AI Research Engineer" is used as a generic industry job title here, not an academic/pure-research signal — the JD's "pure research/academic-only, zero production deployment" archetype was not generated into this dataset.

**Decision:** lock disqualifier #1 as dormant (documented), same treatment as #3 and #5. This is now the third of six hard disqualifiers confirmed absent from the data — a pattern worth calling out explicitly in the Stage 5 defense: three of the six JD-stated disqualifiers are structurally impossible to trigger on this synthetic dataset, which is itself a finding, not a gap in our rule-building.

In [20]:
def hard_disq_research_only_career(row):
    """Entire career research/academic-titled with zero production deployment evidence.
    Confirmed dormant on this dataset (all 32 title-matches have production-deployment
    language in every description; template pool has no academic-only archetype)."""
    all_titles = [row['profile_current_title']] + [j['title'] for j in row['career_history']]
    return all(any(kw in t.lower() for kw in RESEARCH_ACADEMIC_TITLE_KEYWORDS) for t in all_titles)

df_flat['hard_disq_research_only_career'] = df_flat.apply(hard_disq_research_only_career, axis=1)

print("=== Hard disqualifiers — running tally (5 of 6 built) ===")
print("1. Pure research/academic-only, zero production:", df_flat['hard_disq_research_only_career'].sum(), "(dormant, documented)")
print("2. LangChain-wrapper-only, self-learner:", df_flat['hard_disq_langchain_wrapper_only'].sum())
print("3. No production code 18mo+ senior:", df_flat['hard_disq_no_code_18mo_senior'].sum(), "(dormant, documented)")
print("4. Consulting-only career:", df_flat['hard_disq_consulting_only'].sum())
print("5. CV/speech/robotics-only, no NLP/IR:", df_flat['hard_disq_cv_speech_robotics_only'].sum(), "(dormant, documented)")
print("\nRemaining: #6 closed-source-only 5+ yrs, zero external validation (structural, not text-based)")

=== Hard disqualifiers — running tally (5 of 6 built) ===
1. Pure research/academic-only, zero production: 32 (dormant, documented)
2. LangChain-wrapper-only, self-learner: 6
3. No production code 18mo+ senior: 0 (dormant, documented)
4. Consulting-only career: 7034
5. CV/speech/robotics-only, no NLP/IR: 0 (dormant, documented)

Remaining: #6 closed-source-only 5+ yrs, zero external validation (structural, not text-based)


## Step 1t: Hard Disqualifier #6 — Closed-source-only, 5+ yrs, zero external validation

JD wording: 5+ years entirely on closed-source proprietary systems, zero external validation (no papers, talks, OSS). No direct schema field for "papers/talks/OSS," so operationalize with the closest structural proxies available:
- `years_of_experience >= 5`
- `sig_has_github == 0` (no GitHub linked at all — strongest available proxy for zero OSS presence)
- `certifications` count == 0 (no external validation via certs either)
- `verified_skill_count == 0` (no third-party-verified skill assessments — another form of external validation)

Check the funnel stage-by-stage before committing, since every prior text/title-based disqualifier collapsed to near-zero on this dataset.

In [21]:
yoe_ok = df_flat['profile_years_of_experience'] >= 5
print("YOE >= 5:", yoe_ok.sum())

no_github = df_flat['sig_has_github'] == 0
print("...and no GitHub:", (yoe_ok & no_github).sum())

no_certs = df_flat['certifications'].apply(len) == 0
print("...and zero certifications:", (yoe_ok & no_github & no_certs).sum())

no_verified_skills = df_flat['verified_skill_count'] == 0
print("...and zero verified skill assessments:", (yoe_ok & no_github & no_certs & no_verified_skills).sum())

def hard_disq_closed_source_only(row):
    if row['profile_years_of_experience'] < 5:
        return False
    if row['sig_has_github'] != 0:
        return False
    if len(row['certifications']) != 0:
        return False
    if row['verified_skill_count'] != 0:
        return False
    return True

df_flat['hard_disq_closed_source_only'] = df_flat.apply(hard_disq_closed_source_only, axis=1)
print("\nHard disqualifier #6 — closed-source-only, 5+ yrs, zero external validation:", df_flat['hard_disq_closed_source_only'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_closed_source_only']].head(8)
for idx, row in sample.iterrows():
    print(f"{row['candidate_id']} | yoe: {row['profile_years_of_experience']} | title: {row['profile_current_title']} @ {row['profile_current_company']} | industry: {row['profile_current_industry']}")

YOE >= 5: 66130
...and no GitHub: 43369
...and zero certifications: 32610
...and zero verified skill assessments: 26048

Hard disqualifier #6 — closed-source-only, 5+ yrs, zero external validation: 26048

--- Sample flagged candidates ---
CAND_0000002 | yoe: 12.5 | title: Operations Manager @ Wipro | industry: IT Services
CAND_0000005 | yoe: 11.0 | title: Accountant @ Stark Industries | industry: Manufacturing
CAND_0000006 | yoe: 6.0 | title: Business Analyst @ Wayne Enterprises | industry: Conglomerate
CAND_0000007 | yoe: 5.5 | title: Civil Engineer @ Wipro | industry: IT Services
CAND_0000009 | yoe: 11.0 | title: Mechanical Engineer @ Dunder Mifflin | industry: Paper Products
CAND_0000015 | yoe: 5.4 | title: Software Engineer @ Razorpay | industry: Fintech
CAND_0000018 | yoe: 6.6 | title: Frontend Engineer @ Acme Corp | industry: Manufacturing
CAND_0000019 | yoe: 6.5 | title: Project Manager @ Wayne Enterprises | industry: Conglomerate


## Step 1u: Fix — gate disqualifier #6 on genuine AI/ML-relevant technical background first

26,048 flagged is implausibly high and the sample confirms it — Operations Manager, Accountant, Civil Engineer, Mechanical Engineer. This isn't "closed-source AI engineer with no external validation," it's just "no GitHub + no certs," which describes a huge chunk of the non-technical population. The JD's disqualifier only applies to candidates who are otherwise real AI/ML-relevant engineers. Gate on:
- Has at least 2 ML/AI-adjacent skills (`ML_AI_SKILLS` or `NLP_IR_SKILLS` overlap), reusing the taxonomy from earlier steps
- AND still zero GitHub, zero certs, zero verified skills, 5+ yrs — the original structural check

In [22]:
ML_RELEVANT_SKILLS = ML_AI_SKILLS | NLP_IR_SKILLS | CV_SPEECH_ROBOTICS

def has_ml_relevant_background(row):
    skill_names = {s['name'] for s in row['skills']}
    return len(skill_names & ML_RELEVANT_SKILLS) >= 2

df_flat['_has_ml_relevant_background'] = df_flat.apply(has_ml_relevant_background, axis=1)
print("Candidates with 2+ ML/AI-relevant skills:", df_flat['_has_ml_relevant_background'].sum())

def hard_disq_closed_source_only(row):
    if not row['_has_ml_relevant_background']:
        return False
    if row['profile_years_of_experience'] < 5:
        return False
    if row['sig_has_github'] != 0:
        return False
    if len(row['certifications']) != 0:
        return False
    if row['verified_skill_count'] != 0:
        return False
    return True

df_flat['hard_disq_closed_source_only'] = df_flat.apply(hard_disq_closed_source_only, axis=1)
df_flat = df_flat.drop(columns=['_has_ml_relevant_background'])

print("\nHard disqualifier #6 (gated) — ML-relevant, closed-source-only, 5+ yrs, zero external validation:", df_flat['hard_disq_closed_source_only'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_closed_source_only']].head(10)
for idx, row in sample.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} | yoe: {row['profile_years_of_experience']} | title: {row['profile_current_title']} @ {row['profile_current_company']} | skills: {skill_names}")

Candidates with 2+ ML/AI-relevant skills: 29728

Hard disqualifier #6 (gated) — ML-relevant, closed-source-only, 5+ yrs, zero external validation: 3879

--- Sample flagged candidates ---
CAND_0000015 | yoe: 5.4 | title: Software Engineer @ Razorpay | skills: ['PyTorch', 'Content Writing', 'Weights & Biases', 'Qdrant', 'Sales', 'Figma', 'BigQuery', 'Computer Vision', 'Next.js', 'SEO']
CAND_0000032 | yoe: 8.1 | title: .NET Developer @ Cognizant | skills: ['Speech Recognition', 'Project Management', 'REST APIs', 'CSS', 'Embeddings', 'Hadoop', 'Spark', 'Python', 'Data Pipelines', 'OpenCV']
CAND_0000043 | yoe: 8.3 | title: Cloud Engineer @ Swiggy | skills: ['Elasticsearch', 'OpenSearch', 'Airflow', 'Kubeflow', 'Fine-tuning LLMs', 'Haystack', 'OpenCV', 'TensorFlow', 'PEFT', 'LangChain', 'Weights & Biases', 'Reinforcement Learning', 'Deep Learning', 'Image Classification', 'Node.js', 'Project Management', 'Feature Engineering']
CAND_0000051 | yoe: 6.2 | title: Cloud Engineer @ HCL | skills: [

## Step 1v: Fix — gate disqualifier #6 on genuine ML/AI-titled career, not skill-count overlap

Skill-overlap gating (2+ ML-relevant skills) still let through non-technical/adjacent titles (.NET Developer, Cloud Engineer, Graphic Designer) because skills are assigned largely independent of title in this dataset — confirmed earlier during disqualifier #5 debugging. Switch the gate to `is_ml_ai_practitioner_by_title` (the same title-based ML/AI practitioner flag built in Step 1e), which is the reliable signal.

In [23]:
print("Candidates who are genuine ML/AI practitioners by title:", df_flat['is_ml_ai_practitioner_by_title'].sum())

def hard_disq_closed_source_only(row):
    if not row['is_ml_ai_practitioner_by_title']:
        return False
    if row['profile_years_of_experience'] < 5:
        return False
    if row['sig_has_github'] != 0:
        return False
    if len(row['certifications']) != 0:
        return False
    if row['verified_skill_count'] != 0:
        return False
    return True

df_flat['hard_disq_closed_source_only'] = df_flat.apply(hard_disq_closed_source_only, axis=1)

print("\nHard disqualifier #6 (title-gated) — genuine ML/AI practitioner, closed-source-only, 5+ yrs, zero external validation:", df_flat['hard_disq_closed_source_only'].sum())

print("\n--- Sample flagged candidates ---")
sample = df_flat[df_flat['hard_disq_closed_source_only']].head(10)
for idx, row in sample.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} | yoe: {row['profile_years_of_experience']} | title: {row['profile_current_title']} @ {row['profile_current_company']} | skills: {skill_names}")

Candidates who are genuine ML/AI practitioners by title: 988

Hard disqualifier #6 (title-gated) — genuine ML/AI practitioner, closed-source-only, 5+ yrs, zero external validation: 19

--- Sample flagged candidates ---
CAND_0001218 | yoe: 6.4 | title: AI Specialist @ Nykaa | skills: ['ASR', 'pgvector', 'OpenCV', 'scikit-learn', 'Tailwind', 'OpenSearch', 'NLP', 'SQL', 'LlamaIndex', 'Accounting', 'Object Detection', 'Speech Recognition', 'RAG', 'Semantic Search', 'Data Science', 'TTS', 'PowerPoint', 'QLoRA']
CAND_0003290 | yoe: 5.9 | title: Computer Vision Engineer @ Verloop.io | skills: ['QLoRA', 'Pinecone', 'YOLO', 'Feature Engineering', 'Weights & Biases', 'Forecasting', 'LlamaIndex', 'Illustrator', 'GANs', 'Machine Learning', 'LangChain', 'Diffusion Models', 'TensorFlow', 'OpenCV', 'Weaviate', 'BentoML']
CAND_0007052 | yoe: 6.5 | title: ML Engineer @ Unacademy | skills: ['Time Series', 'Data Science', 'ASR', 'RAG', 'Statistical Modeling', 'Diffusion Models', 'Deep Learning', 'Weaviat

## Step 1w: Confirmed — disqualifier #6 is a real, non-dormant population

19 candidates: genuine ML/AI-titled (988-candidate pool), 5+ yrs experience, rich relevant skill sets, but zero GitHub, zero certifications, zero verified skill assessments — a real "strong-looking on paper, zero external validation trail" pattern. This matches the JD's intent precisely. Lock in as final.

**All six hard disqualifiers now built:**
| # | Disqualifier | Count | Status |
|---|---|---|---|
| 1 | Pure research/academic-only | 32 | dormant (false positives on inspection — not a real pattern) |
| 2 | LangChain-wrapper-only, self-learner | 6 | active |
| 3 | No production code 18mo+ senior | 0 | dormant |
| 4 | Consulting-only career | 7,034 | active |
| 5 | CV/speech/robotics-only, no NLP/IR | 0 | dormant |
| 6 | Closed-source-only, zero external validation | 19 | active |

Note: disqualifier #1's title-match (32) was later found to be false positives on inspection (Step 1s) — every one had production-deployment description text — so it's set to always return `False` going forward, same as #3 and #5. Only #2, #4, #6 are live, non-trivial disqualifiers on this dataset.

In [24]:
df_flat['is_hard_disqualified'] = (
    df_flat['hard_disq_research_only_career'] |       # dormant, always False
    df_flat['hard_disq_langchain_wrapper_only'] |
    df_flat['hard_disq_no_code_18mo_senior'] |          # dormant, always False
    df_flat['hard_disq_consulting_only'] |
    df_flat['hard_disq_cv_speech_robotics_only'] |      # dormant, always False
    df_flat['hard_disq_closed_source_only']
)

print("=== FINAL: Hard disqualifier tally ===")
print("1. Research-only (dormant):", df_flat['hard_disq_research_only_career'].sum())
print("2. LangChain-wrapper-only:", df_flat['hard_disq_langchain_wrapper_only'].sum())
print("3. No-code-18mo-senior (dormant):", df_flat['hard_disq_no_code_18mo_senior'].sum())
print("4. Consulting-only:", df_flat['hard_disq_consulting_only'].sum())
print("5. CV/speech/robotics-only (dormant):", df_flat['hard_disq_cv_speech_robotics_only'].sum())
print("6. Closed-source-only:", df_flat['hard_disq_closed_source_only'].sum())
print("\nTOTAL candidates hard-disqualified (any flag):", df_flat['is_hard_disqualified'].sum(), "/", len(df_flat))

# Save NB-02 checkpoint
import os
from pathlib import Path
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

disq_cols = ['candidate_id', 'hard_disq_research_only_career', 'hard_disq_langchain_wrapper_only',
             'hard_disq_no_code_18mo_senior', 'hard_disq_consulting_only',
             'hard_disq_cv_speech_robotics_only', 'hard_disq_closed_source_only', 'is_hard_disqualified']

df_flat[disq_cols].to_parquet(OUT_DIR / "hard_disqualifier_flags_v1.parquet", index=False)
print("\nSaved hard_disqualifier_flags_v1.parquet")
print("Shape:", df_flat[disq_cols].shape)

=== FINAL: Hard disqualifier tally ===
1. Research-only (dormant): 32
2. LangChain-wrapper-only: 6
3. No-code-18mo-senior (dormant): 0
4. Consulting-only: 7034
5. CV/speech/robotics-only (dormant): 0
6. Closed-source-only: 19

TOTAL candidates hard-disqualified (any flag): 7089 / 100000

Saved hard_disqualifier_flags_v1.parquet
Shape: (100000, 8)


## Step 1x: Decision — exclude disqualifier #1 from the master flag (confirmed false-positive, not dormant)

Disqualifier #3 and #5 are genuinely **dormant** (0 matches — the pattern doesn't exist in the data). Disqualifier #1 is different: it **does** match 32 candidates, but Step 1s confirmed every match is a false positive (production ML engineers whose title happens to be "AI Research Engineer" at every role). Including it in `is_hard_disqualified` would incorrectly disqualify 32 legitimate candidates. Correct this now, before it propagates into ranking.

In [25]:
# Rebuild master flag, excluding the confirmed-false-positive disqualifier #1
df_flat['is_hard_disqualified'] = (
    df_flat['hard_disq_langchain_wrapper_only'] |
    df_flat['hard_disq_consulting_only'] |
    df_flat['hard_disq_closed_source_only']
    # hard_disq_research_only_career EXCLUDED — confirmed false-positive-only on this dataset (Step 1s)
    # hard_disq_no_code_18mo_senior and hard_disq_cv_speech_robotics_only excluded — dormant, always False anyway
)

print("=== FINAL (corrected): Hard disqualifier tally ===")
print("2. LangChain-wrapper-only:", df_flat['hard_disq_langchain_wrapper_only'].sum())
print("4. Consulting-only:", df_flat['hard_disq_consulting_only'].sum())
print("6. Closed-source-only:", df_flat['hard_disq_closed_source_only'].sum())
print("\nTOTAL candidates hard-disqualified (corrected):", df_flat['is_hard_disqualified'].sum(), "/", len(df_flat))

# Re-save corrected checkpoint
disq_cols = ['candidate_id', 'hard_disq_research_only_career', 'hard_disq_langchain_wrapper_only',
             'hard_disq_no_code_18mo_senior', 'hard_disq_consulting_only',
             'hard_disq_cv_speech_robotics_only', 'hard_disq_closed_source_only', 'is_hard_disqualified']

df_flat[disq_cols].to_parquet(OUT_DIR / "hard_disqualifier_flags_v1.parquet", index=False)
print("\nRe-saved hard_disqualifier_flags_v1.parquet (corrected)")

=== FINAL (corrected): Hard disqualifier tally ===
2. LangChain-wrapper-only: 6
4. Consulting-only: 7034
6. Closed-source-only: 19

TOTAL candidates hard-disqualified (corrected): 7059 / 100000

Re-saved hard_disqualifier_flags_v1.parquet (corrected)


## Step 2a: Soft Negative #1 — Title-chasing pattern

JD wording: Senior→Staff→Principal (or equivalent escalating-seniority titles) via company-hops of roughly ~1.5 years each. This is a down-weight signal, not a disqualifier — it suggests title inflation via job-hopping rather than depth built through sustained ownership.

Operationalize as:
- career_history has 2+ entries
- Consecutive entries show strictly increasing seniority rank (using the same `SENIORITY_TITLE_MARKERS`-style ordinal: IC < Senior < Staff/Lead < Principal/Director/Head)
- Each of those escalating entries has `duration_months` roughly in the 12-20 month range (short tenure, not long-term ownership)

Output a continuous score (count of qualifying escalation-hops), not just a boolean — soft negatives should scale the down-weight, not gate pass/fail.

In [26]:
SENIORITY_RANK = {
    'principal': 5, 'director': 5, 'head of': 5,
    'staff': 4, 'architect': 4,
    'lead': 3, 'senior': 3,
    # unranked / IC-level titles default to 1 via .get() fallback
}

def seniority_rank(title):
    title_lower = title.lower()
    for marker, rank in sorted(SENIORITY_RANK.items(), key=lambda x: -x[1]):
        if marker in title_lower:
            return rank
    return 1  # base IC level, no seniority marker

def title_chasing_score(career_history, min_months=12, max_months=20):
    if len(career_history) < 2:
        return 0
    # sort chronologically by start_date
    sorted_jobs = sorted(career_history, key=lambda j: j['start_date'])
    hops = 0
    for i in range(len(sorted_jobs) - 1):
        cur_rank = seniority_rank(sorted_jobs[i]['title'])
        next_rank = seniority_rank(sorted_jobs[i+1]['title'])
        cur_duration = sorted_jobs[i].get('duration_months', 0)
        if next_rank > cur_rank and min_months <= cur_duration <= max_months:
            hops += 1
    return hops

df_flat['soft_neg_title_chasing_score'] = df_flat['career_history'].apply(title_chasing_score)
print("title_chasing_score distribution:")
print(df_flat['soft_neg_title_chasing_score'].value_counts().sort_index())

print("\n--- Sample candidates with score >= 1 ---")
sample = df_flat[df_flat['soft_neg_title_chasing_score'] >= 1].head(8)
for idx, row in sample.iterrows():
    jobs = sorted(row['career_history'], key=lambda j: j['start_date'])
    titles = [(j['title'], j['duration_months']) for j in jobs]
    print(f"{row['candidate_id']} | score: {row['soft_neg_title_chasing_score']} | path: {titles}")

title_chasing_score distribution:
soft_neg_title_chasing_score
0    99551
1      447
2        2
Name: count, dtype: int64

--- Sample candidates with score >= 1 ---
CAND_0001025 | score: 1 | path: [('Data Engineer', 16), ('Senior Software Engineer', 22), ('Software Engineer', 24), ('Senior Software Engineer', 31)]
CAND_0001158 | score: 1 | path: [('Data Analyst', 18), ('Senior Data Engineer', 24), ('Data Engineer', 40)]
CAND_0001218 | score: 1 | path: [('Computer Vision Engineer', 22), ('Computer Vision Engineer', 18), ('Senior Software Engineer (ML)', 14), ('AI Specialist', 21)]
CAND_0001475 | score: 1 | path: [('Data Engineer', 18), ('Senior Data Engineer', 18), ('Data Engineer', 33)]
CAND_0001575 | score: 1 | path: [('Data Engineer', 12), ('Senior Software Engineer', 36), ('Data Engineer', 28)]
CAND_0001968 | score: 1 | path: [('Data Engineer', 12), ('Senior Data Engineer', 40)]
CAND_0002028 | score: 1 | path: [('Data Analyst', 22), ('Analytics Engineer', 22), ('Software Engineer', 

## Step 2b: Soft Negative #2 — Framework-tutorial signature

JD wording: "framework enthusiast" pattern — skills/profile dominated by tutorial-style LangChain/hot-framework projects rather than systems work. This overlaps conceptually with hard disqualifier #2 but is a *softer* version: candidates who lean heavily on trendy framework tags (LangChain, LlamaIndex, Prompt Engineering, RAG) relative to their systems/infra depth, without meeting the full disqualifier #2 bar (recency + zero real ML production skill).

Operationalize as a continuous ratio, not boolean:
- `framework_tag_count` = count of trendy/wrapper skill tags present (`LLM_WRAPPER_SKILLS`, reused from disqualifier #2)
- `systems_depth_count` = count of systems/infra-signaling skills present (MLOps, Docker, Kubernetes, distributed systems, vector DB infra, etc. — reuse `GENERAL_ML_INFRA`)
- `framework_tutorial_ratio` = framework_tag_count / (framework_tag_count + systems_depth_count + 1), so it's bounded [0, ~1) and defined even when systems_depth_count is 0

In [27]:
def framework_tutorial_ratio(skills_list):
    skill_names = {s['name'] for s in skills_list}
    framework_count = len(skill_names & LLM_WRAPPER_SKILLS)
    systems_count = len(skill_names & GENERAL_ML_INFRA)
    return framework_count / (framework_count + systems_count + 1)

df_flat['soft_neg_framework_tutorial_ratio'] = df_flat['skills'].apply(framework_tutorial_ratio)

print("framework_tutorial_ratio distribution:")
print(df_flat['soft_neg_framework_tutorial_ratio'].describe())

print("\n--- Candidates with highest ratio (only among genuine ML/AI-titled, to avoid the earlier skill-noise trap) ---")
ml_titled_sorted = df_flat[df_flat['is_ml_ai_practitioner_by_title']].sort_values('soft_neg_framework_tutorial_ratio', ascending=False).head(10)
for idx, row in ml_titled_sorted.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} | ratio: {row['soft_neg_framework_tutorial_ratio']:.2f} | title: {row['profile_current_title']} | skills: {skill_names}")

framework_tutorial_ratio distribution:
count    100000.000000
mean          0.049372
std           0.158361
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.750000
Name: soft_neg_framework_tutorial_ratio, dtype: float64

--- Candidates with highest ratio (only among genuine ML/AI-titled, to avoid the earlier skill-noise trap) ---
CAND_0030827 | ratio: 0.60 | title: Senior Data Scientist | skills: ['Hugging Face Transformers', 'NLP', 'Recommendation Systems', 'FAISS', 'Reinforcement Learning', 'CSS', 'LlamaIndex', 'TTS', 'Qdrant', 'BM25', 'Sentence Transformers', 'Semantic Search', 'Object Detection', 'Learning to Rank', 'LangChain', 'Prompt Engineering']
CAND_0000200 | ratio: 0.60 | title: ML Engineer | skills: ['BigQuery', 'Vector Search', 'Prompt Engineering', 'BentoML', 'LlamaIndex', 'FAISS', 'RAG', 'Object Detection', 'Kubernetes', 'TTS', 'Kubeflow', 'Apache Flink']
CAND_0082910 | ratio: 0.57 | title: AI Specialist | skills

## Step 2c: Fix — framework_tutorial_ratio conflates "no infra tags" with "no depth"

The current denominator (`GENERAL_ML_INFRA`) only counts infra/MLOps tooling skills, not genuine modeling depth. Several high-ratio candidates (CAND_0030827, CAND_0082910, CAND_0040200) have real depth — FAISS, PyTorch, CNN, GANs, Diffusion Models — they just don't happen to have Docker/Kubernetes/MLOps tags. That's not the "framework tutorial, no systems work" pattern the JD describes.

**Fix:** widen the "depth" side of the ratio to include genuine modeling/NLP/CV skills (`NLP_IR_SKILLS | CV_SPEECH_ROBOTICS | ML_AI_SKILLS`), not just infra tooling — a candidate is only "framework-tutorial-signature" if their LLM-wrapper tags dominate with **no supporting modeling depth of any kind**, not merely a lack of specific infra tags.

In [28]:
GENUINE_ML_DEPTH_SKILLS = (NLP_IR_SKILLS | CV_SPEECH_ROBOTICS | ML_AI_SKILLS | GENERAL_ML_INFRA) - LLM_WRAPPER_SKILLS

def framework_tutorial_ratio(skills_list):
    skill_names = {s['name'] for s in skills_list}
    framework_count = len(skill_names & LLM_WRAPPER_SKILLS)
    depth_count = len(skill_names & GENUINE_ML_DEPTH_SKILLS)
    return framework_count / (framework_count + depth_count + 1)

df_flat['soft_neg_framework_tutorial_ratio'] = df_flat['skills'].apply(framework_tutorial_ratio)

print("framework_tutorial_ratio distribution (corrected):")
print(df_flat['soft_neg_framework_tutorial_ratio'].describe())

print("\n--- Highest-ratio genuine ML/AI-titled candidates (corrected) ---")
ml_titled_sorted = df_flat[df_flat['is_ml_ai_practitioner_by_title']].sort_values('soft_neg_framework_tutorial_ratio', ascending=False).head(10)
for idx, row in ml_titled_sorted.iterrows():
    skill_names = [s['name'] for s in row['skills']]
    print(f"{row['candidate_id']} | ratio: {row['soft_neg_framework_tutorial_ratio']:.2f} | title: {row['profile_current_title']} | skills: {skill_names}")

framework_tutorial_ratio distribution (corrected):
count    100000.000000
mean          0.020804
std           0.068395
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.666667
Name: soft_neg_framework_tutorial_ratio, dtype: float64

--- Highest-ratio genuine ML/AI-titled candidates (corrected) ---
CAND_0000200 | ratio: 0.33 | title: ML Engineer | skills: ['BigQuery', 'Vector Search', 'Prompt Engineering', 'BentoML', 'LlamaIndex', 'FAISS', 'RAG', 'Object Detection', 'Kubernetes', 'TTS', 'Kubeflow', 'Apache Flink']
CAND_0040117 | ratio: 0.31 | title: Recommendation Systems Engineer | skills: ['OpenCV', 'LlamaIndex', 'Embeddings', 'RAG', 'FAISS', 'BM25', 'Prompt Engineering', 'LoRA', 'Diffusion Models', 'LangChain', 'Reinforcement Learning', 'QLoRA']
CAND_0082910 | ratio: 0.31 | title: AI Specialist | skills: ['Diffusion Models', 'Speech Recognition', 'BigQuery', 'LlamaIndex', 'OpenCV', 'GCP', 'Kafka', 'Sentence Transformers', 'GA

## Step 3: Finalize NB-02 — combine hard disqualifiers + soft negatives, save output

Close out this notebook: consolidate all rule columns (3 active hard disqualifiers, 3 dormant-but-documented, 2 soft-negative continuous scores) into a single output parquet for NB-03 (Feature Engineering) and NB-05 (Pseudo-Label/Tier Assignment) to consume.

In [29]:
JD_RULE_COLUMNS = [
    'candidate_id',
    # Hard disqualifiers — active
    'hard_disq_langchain_wrapper_only',
    'hard_disq_consulting_only',
    'hard_disq_closed_source_only',
    # Hard disqualifiers — dormant, documented (kept for completeness/defensibility)
    'hard_disq_research_only_career',
    'hard_disq_no_code_18mo_senior',
    'hard_disq_cv_speech_robotics_only',
    # Master gate (excludes confirmed false-positive #1, per Step 1x)
    'is_hard_disqualified',
    # Soft negatives — continuous scores
    'soft_neg_title_chasing_score',
    'soft_neg_framework_tutorial_ratio',
]

jd_rule_flags = df_flat[JD_RULE_COLUMNS].copy()

print("jd_rule_flags shape:", jd_rule_flags.shape)
print("\n--- Summary ---")
print("Hard disqualified (final, active rules only):", jd_rule_flags['is_hard_disqualified'].sum())
print("Soft neg — title chasing (score >= 1):", (jd_rule_flags['soft_neg_title_chasing_score'] >= 1).sum())
print("Soft neg — framework tutorial ratio (> 0.2):", (jd_rule_flags['soft_neg_framework_tutorial_ratio'] > 0.2).sum())

jd_rule_flags.to_parquet(OUT_DIR / "jd_rule_flags.parquet", index=False)
print("\nSaved jd_rule_flags.parquet")
print("Size on disk:", (OUT_DIR / "jd_rule_flags.parquet").stat().st_size / 1024, "KB")

jd_rule_flags shape: (100000, 10)

--- Summary ---
Hard disqualified (final, active rules only): 7059
Soft neg — title chasing (score >= 1): 449
Soft neg — framework tutorial ratio (> 0.2): 4351

Saved jd_rule_flags.parquet
Size on disk: 672.3134765625 KB
